In [ ]:
# ============================================================
# AI Innovators Lab: Heart Disease Risk Prediction
# ============================================================
# This notebook teaches students how an AI model can learn patterns from data.
# The example topic is heart disease risk prediction.
# The goal is to understand the AI process, not to make real medical decisions.
# This project uses a real public dataset, but students will only test fictional examples.
# IMPORTANT: This notebook is for education only.
# IMPORTANT: This model is NOT medical advice.
# IMPORTANT: This model should NOT be used to diagnose anyone.
# Dataset source: UCI Heart Disease Dataset.
# Dataset link: https://archive.ics.uci.edu/dataset/45/heart+disease
# Model used in this notebook: Random Forest Classifier.
# A Random Forest is a group of decision trees that vote together.

# Display a welcome message so students know the project has started.
print("Welcome to the Heart Disease Prediction Project!")

# Explain the main learning goal in a simple sentence.
print("Goal: Learn how AI can use health-related data to make predictions.")

# Remind students that this is only a classroom activity.
print("Important: This is for education only, not medical advice.")


In [ ]:
# ============================================================
# Step 1: Import required libraries
# ============================================================
# A library is a collection of prewritten code that helps us do common tasks.
# Instead of writing everything from scratch, we import libraries.

# Import pandas and give it the short nickname pd.
# pandas helps us work with tables of data, similar to spreadsheets.
import pandas as pd

# Import NumPy and give it the short nickname np.
# NumPy helps Python work with numbers and missing values.
import numpy as np

# Import matplotlib's plotting tool and give it the short nickname plt.
# matplotlib helps us draw charts and graphs.
import matplotlib.pyplot as plt

# Import requests.
# requests helps Python download files from the internet.
import requests

# Import os.
# os helps Python work with files and folders on the computer.
import os

# Import train_test_split from scikit-learn.
# This helps us divide the data into training data and testing data.
from sklearn.model_selection import train_test_split

# Import RandomForestClassifier.
# This is the machine learning model we will train.
from sklearn.ensemble import RandomForestClassifier

# Import accuracy_score.
# This helps us measure how often the model made the correct prediction.
from sklearn.metrics import accuracy_score

# Import confusion_matrix.
# This helps us see what types of mistakes the model made.
from sklearn.metrics import confusion_matrix

# Import classification_report.
# This gives a more detailed performance summary for the model.
from sklearn.metrics import classification_report

# Print a success message after all libraries are imported.
print("Libraries loaded successfully!")


In [ ]:
# ============================================================
# Step 2: Download the UCI Heart Disease dataset
# ============================================================
# In this step, Python downloads the dataset from the internet.
# The dataset is a table where each row represents one patient record.

# Store the web address of the dataset in a variable.
# A variable is like a labeled box that stores information.
dataset_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

# Store the file name we want to use when saving the downloaded dataset.
file_name = "processed_cleveland_heart_disease.csv"

# Use requests.get() to download the dataset from the web address.
# The downloaded result is stored in the variable named response.
response = requests.get(dataset_url)

# Open a new file on the computer using write-binary mode, written as "wb".
# "wb" is used because the downloaded content comes in raw file form.
with open(file_name, "wb") as file:

    # Write the downloaded dataset content into the file.
    file.write(response.content)

# Print a message so students know the download step worked.
print("Dataset downloaded successfully!")


In [ ]:
# ============================================================
# Step 3: Load the dataset
# ============================================================
# Loading means reading the saved file into Python.
# The dataset file does not include column names, so we create names ourselves.
# Column names help us understand what each piece of data means.

# Create a list of column names.
# A list stores multiple items in order.
columns = [

    # Patient age.
    "age",

    # Patient sex in the original dataset format.
    "sex",

    # Type of chest pain recorded in the dataset.
    "chest_pain_type",

    # Resting blood pressure.
    "resting_blood_pressure",

    # Cholesterol value.
    "cholesterol",

    # Whether fasting blood sugar is above a certain level.
    "fasting_blood_sugar",

    # Resting electrocardiogram result.
    "resting_ecg",

    # Maximum heart rate reached.
    "max_heart_rate",

    # Whether exercise caused chest pain.
    "exercise_angina",

    # ST depression value from the medical test.
    "oldpeak",

    # Slope of the peak exercise ST segment.
    "slope",

    # Number of major vessels colored by fluoroscopy.
    "ca",

    # Thalassemia related test result.
    "thal",

    # Original target value from the dataset.
    "target"
]

# Read the CSV file into a pandas DataFrame named data.
# A DataFrame is a table with rows and columns.
# names=columns tells pandas to use the column names we created above.
data = pd.read_csv(file_name, names=columns)

# Print a message so students know the dataset loaded correctly.
print("Dataset loaded successfully!")

# Show the first five rows of the dataset.
# This helps students see what the data looks like.
print(data.head())


In [ ]:
# ============================================================
# Step 4: Clean the dataset
# ============================================================
# Real datasets can contain missing or messy values.
# Cleaning means fixing or removing values that Python cannot use properly.
# In this dataset, some missing values are written as question marks: "?"

# Replace every "?" in the table with np.nan.
# np.nan means "missing value" in NumPy and pandas.
data = data.replace("?", np.nan)

# Remove rows that contain missing values.
# This keeps the notebook simple for beginner students.
data = data.dropna()

# Go through every column name in the dataset one by one.
# A for loop repeats the same action for each item.
for col in data.columns:

    # Convert the current column to numbers.
    # Machine learning models need numbers, not text.
    data[col] = pd.to_numeric(data[col])

# Print a message after the cleaning is finished.
print("Dataset cleaned successfully!")

# Print how many rows remain after removing incomplete rows.
print("Number of rows after cleaning:", len(data))


In [ ]:
# ============================================================
# Step 5: Create simple labels
# ============================================================
# The original dataset uses target values from 0 to 4.
# target = 0 means no heart disease was found.
# target = 1, 2, 3, or 4 means some level of heart disease was present.
# To make the project easier, we convert the target into two groups:
# 0 = Lower Risk
# 1 = Higher Risk

# Create a new column named risk.
# apply() runs a small rule on every value in the target column.
# lambda x means "for each target value, call it x."
# If x is 0, the new risk value becomes 0.
# If x is not 0, the new risk value becomes 1.
data["risk"] = data["target"].apply(lambda x: 0 if x == 0 else 1)

# Print a message so students know the new labels were created.
print("Risk labels created!")

# Count how many examples are Lower Risk and Higher Risk.
# This helps students see whether the dataset is balanced.
print(data["risk"].value_counts())


In [ ]:
# ============================================================
# Step 6: Select easy-to-understand input features
# ============================================================
# Features are the input information the model uses to make predictions.
# We select only four features so the model is easier for students to understand.
# The selected features are age, resting blood pressure, cholesterol, and max heart rate.

# Create X, which stores the input features.
# X is commonly used in machine learning to mean "inputs."
# The double brackets select several columns from the DataFrame.
X = data[[

    # Use age as one input feature.
    "age",

    # Use resting blood pressure as one input feature.
    "resting_blood_pressure",

    # Use cholesterol as one input feature.
    "cholesterol",

    # Use maximum heart rate as one input feature.
    "max_heart_rate"
]]

# Create y, which stores the answer the model is trying to learn.
# y is commonly used in machine learning to mean "target" or "label."
# Here, y tells whether each row is Lower Risk or Higher Risk.
y = data["risk"]

# Print a heading before showing the selected input features.
print("Selected input features:")

# Show the first five rows of X.
# This helps students see what information the model will study.
print(X.head())


In [ ]:
# ============================================================
# Step 7: Split data into training and testing sets
# ============================================================
# The model should not be tested on the exact same examples it studied.
# Training data is used to teach the model.
# Testing data is used to check how well the model learned.
# This is similar to studying with practice questions and then taking a quiz.

# Split the input features and labels into training and testing groups.
X_train, X_test, y_train, y_test = train_test_split(

    # X contains the input features.
    X,

    # y contains the correct answers.
    y,

    # test_size=0.25 means 25% of the data will be used for testing.
    test_size=0.25,

    # random_state=42 makes the split repeatable.
    # This means students get the same result each time they run the notebook.
    random_state=42,

    # stratify=y keeps the Lower Risk and Higher Risk ratio similar in both groups.
    stratify=y
)

# Print the number of examples used for training.
print("Training examples:", len(X_train))

# Print the number of examples used for testing.
print("Testing examples:", len(X_test))


In [ ]:
# ============================================================
# Step 8: Build and train the Random Forest model
# ============================================================
# A Random Forest is made of many decision trees.
# A decision tree asks yes/no style questions about the data.
# Example: Is cholesterol greater than a certain value?
# The forest combines many trees and lets them vote on the final prediction.

# Create the Random Forest model and store it in the variable named model.
model = RandomForestClassifier(

    # n_estimators=100 means the forest will use 100 decision trees.
    # More trees often make the model more stable.
    n_estimators=100,

    # max_depth=5 limits how deep each tree can grow.
    # This helps prevent the model from memorizing the training data too much.
    max_depth=5,

    # random_state=42 makes the model's random choices repeatable.
    random_state=42
)

# Train the model using the training inputs and training answers.
# fit() is the command that teaches the model from data.
model.fit(X_train, y_train)

# Print a message after training is complete.
print("Model training completed!")


In [ ]:
# ============================================================
# Step 9: Evaluate the model
# ============================================================
# Evaluating means checking how well the trained model performs.
# We ask the model to predict the answers for the testing data.
# Then we compare the model's predictions with the real answers.

# Use the trained model to predict labels for the test input data.
# The predictions are stored in y_pred.
y_pred = model.predict(X_test)

# Compare the true answers y_test with the predicted answers y_pred.
# accuracy_score returns the fraction of predictions that were correct.
accuracy = accuracy_score(y_test, y_pred)

# Print the accuracy as a percentage.
# Multiplying by 100 changes a decimal like 0.75 into 75%.
# round(..., 2) keeps only two decimal places.
print("Model Accuracy:", round(accuracy * 100, 2), "%")

# Print a blank line and a heading for the classification report.
print("\nClassification Report:")

# Print precision, recall, and F1-score for both classes.
# Lower Risk means class 0.
# Higher Risk means class 1.
print(classification_report(

    # y_test contains the true answers.
    y_test,

    # y_pred contains the model's predicted answers.
    y_pred,

    # target_names gives friendly names to class 0 and class 1.
    target_names=["Lower Risk", "Higher Risk"]
))

# Print a blank line and a heading for the confusion matrix.
print("\nConfusion Matrix:")

# Print the confusion matrix.
# This shows correct predictions and mistakes in a small table.
print(confusion_matrix(y_test, y_pred))


In [ ]:
# ============================================================
# Step 10: Visualize confusion matrix
# ============================================================
# A confusion matrix shows where the model was correct and where it was wrong.
# Rows usually represent the true labels.
# Columns usually represent the predicted labels.

# Create the confusion matrix using the true answers and predicted answers.
cm = confusion_matrix(y_test, y_pred)

# Create a new figure for the plot.
# figsize=(5, 4) controls the width and height of the chart.
plt.figure(figsize=(5, 4))

# Display the confusion matrix as an image.
# Darker or lighter squares represent different counts.
plt.imshow(cm)

# Add a title to the chart.
plt.title("Confusion Matrix")

# Add a color bar that explains the color scale.
plt.colorbar()

# Label the x-axis positions.
# These are the model's predicted labels.
plt.xticks([0, 1], ["Lower Risk", "Higher Risk"])

# Label the y-axis positions.
# These are the true labels from the dataset.
plt.yticks([0, 1], ["Lower Risk", "Higher Risk"])

# Add a label under the x-axis.
plt.xlabel("Predicted Label")

# Add a label beside the y-axis.
plt.ylabel("True Label")

# Loop through the two row positions: 0 and 1.
for i in range(2):

    # Loop through the two column positions: 0 and 1.
    for j in range(2):

        # Write the number from the confusion matrix inside each square.
        # j controls the horizontal position.
        # i controls the vertical position.
        # ha and va center the text inside the square.
        plt.text(j, i, cm[i, j], ha="center", va="center")

# Show the completed chart.
plt.show()


In [ ]:
# ============================================================
# Step 11: Feature importance
# ============================================================
# Feature importance helps us understand what the model used most.
# It does not prove medical cause.
# It only shows which input features were more useful for this trained model.

# Create a new DataFrame to store each feature name and its importance score.
feature_importance = pd.DataFrame({

    # Store the names of the input features.
    "Feature": X.columns,

    # Store the importance scores learned by the Random Forest model.
    "Importance": model.feature_importances_
})

# Sort the features from most important to least important.
feature_importance = feature_importance.sort_values(

    # Sort using the Importance column.
    by="Importance",

    # ascending=False means largest values come first.
    ascending=False
)

# Print a heading for the feature importance table.
print("Feature Importance:")

# Print the sorted feature importance table.
print(feature_importance)

# Create a new figure for the feature importance chart.
plt.figure(figsize=(8, 5))

# Draw a horizontal bar chart.
# The feature names appear on one axis.
# The importance scores determine the bar lengths.
plt.barh(feature_importance["Feature"], feature_importance["Importance"])

# Add a title to the chart.
plt.title("Which Features Helped the Model Most?")

# Add a label under the x-axis.
plt.xlabel("Importance")

# Add a label beside the y-axis.
plt.ylabel("Feature")

# Reverse the y-axis so the most important feature appears at the top.
plt.gca().invert_yaxis()

# Show the completed chart.
plt.show()


In [ ]:
# ============================================================
# Step 12: Test fictional/random patient examples
# ============================================================
# These examples are NOT real people.
# Students should not enter their own personal health information.
# We use fictional examples to safely see how the model makes predictions.

# Create a DataFrame containing three fictional patient examples.
# Each dictionary inside the list represents one fictional patient.
fictional_patients = pd.DataFrame([

    # Fictional patient 1.
    {
        # Fictional age value.
        "age": 35,

        # Fictional resting blood pressure value.
        "resting_blood_pressure": 120,

        # Fictional cholesterol value.
        "cholesterol": 180,

        # Fictional maximum heart rate value.
        "max_heart_rate": 170
    },

    # Fictional patient 2.
    {
        # Fictional age value.
        "age": 55,

        # Fictional resting blood pressure value.
        "resting_blood_pressure": 145,

        # Fictional cholesterol value.
        "cholesterol": 250,

        # Fictional maximum heart rate value.
        "max_heart_rate": 135
    },

    # Fictional patient 3.
    {
        # Fictional age value.
        "age": 70,

        # Fictional resting blood pressure value.
        "resting_blood_pressure": 165,

        # Fictional cholesterol value.
        "cholesterol": 290,

        # Fictional maximum heart rate value.
        "max_heart_rate": 110
    }
])

# Ask the trained model to predict Lower Risk or Higher Risk for each fictional patient.
fictional_predictions = model.predict(fictional_patients)

# Ask the model for prediction probabilities.
# These probabilities show how confident the model is for each class.
fictional_probabilities = model.predict_proba(fictional_patients)

# Print a heading before showing the results.
print("Fictional Patient Predictions:\n")

# Loop through each fictional patient.
# range(len(fictional_patients)) gives the row numbers 0, 1, and 2.
for i in range(len(fictional_patients)):

    # Get the model's prediction for the current fictional patient.
    prediction = fictional_predictions[i]

    # Get the largest probability for the current prediction.
    # Multiplying by 100 changes it into a percentage.
    confidence = max(fictional_probabilities[i]) * 100

    # Convert the numeric prediction into a friendly text label.
    # If prediction is 1, the label is Higher Risk.
    # Otherwise, the label is Lower Risk.
    label = "Higher Risk" if prediction == 1 else "Lower Risk"

    # Print the fictional patient number.
    # We use i + 1 because people usually count from 1, not 0.
    print("Fictional Patient", i + 1)

    # Print the input values for the current fictional patient.
    print(fictional_patients.iloc[i])

    # Print the AI model's predicted label.
    print("AI Prediction:", label)

    # Print the model's confidence rounded to two decimal places.
    print("Confidence:", round(confidence, 2), "%")

    # Print a line to separate each fictional patient's results.
    print("-" * 60)


In [ ]:
# ============================================================
# Step 13: Let students create a fictional patient
# ============================================================
# Reminder:
# Use made-up or random values only.
# Do NOT enter personal or family medical information.
# This section helps students test how changing inputs changes the prediction.

# Print an instruction message.
print("Create a fictional patient profile.")

# Print a safety reminder.
print("Do NOT enter your own personal health information.")

# Ask the student to type a fictional age.
# input() gets text from the keyboard.
# int() converts the typed text into a whole number.
age = int(input("Enter fictional age, e.g., 45: "))

# Ask the student to type a fictional resting blood pressure value.
# The result is converted to a whole number.
bp = int(input("Enter fictional resting blood pressure, e.g., 130: "))

# Ask the student to type a fictional cholesterol value.
# The result is converted to a whole number.
cholesterol = int(input("Enter fictional cholesterol, e.g., 220: "))

# Ask the student to type a fictional maximum heart rate value.
# The result is converted to a whole number.
max_hr = int(input("Enter fictional max heart rate, e.g., 150: "))

# Create a one-row DataFrame for the fictional example.
# The column names must match the feature names used during training.
fictional_example = pd.DataFrame([{

    # Store the fictional age value.
    "age": age,

    # Store the fictional resting blood pressure value.
    "resting_blood_pressure": bp,

    # Store the fictional cholesterol value.
    "cholesterol": cholesterol,

    # Store the fictional maximum heart rate value.
    "max_heart_rate": max_hr
}])

# Ask the model to predict the risk class for this one fictional example.
# [0] gets the first and only prediction from the result.
prediction = model.predict(fictional_example)[0]

# Ask the model for the probability of each possible class.
# [0] gets the first and only probability row.
probability = model.predict_proba(fictional_example)[0]

# Check whether the model predicted class 1.
# Class 1 means Higher Risk in this notebook.
if prediction == 1:

    # Print the Higher Risk prediction.
    print("\nThe AI predicts: Higher Risk")

# If prediction is not 1, then it must be class 0.
else:

    # Print the Lower Risk prediction.
    print("\nThe AI predicts: Lower Risk")

# Print the model's confidence.
# max(probability) chooses the larger probability value.
# Multiplying by 100 converts it to a percentage.
# round(..., 2) keeps two decimal places.
print("Confidence:", round(max(probability) * 100, 2), "%")

# Print a final safety reminder.
print("\nReminder: This prediction is for education only and is not medical advice.")


In [ ]:
# ============================================================
# Step 14: Save the model
# ============================================================
# Saving the model lets us reuse it later without training again.
# We use pickle to save the trained Python model as a file.

# Import pickle.
# pickle is a Python tool for saving Python objects to files.
import pickle

# Open a file where the trained model will be saved.
# "wb" means write-binary mode.
with open("heart_disease_prediction_model.pkl", "wb") as file:

    # Save the trained model into the file.
    pickle.dump(model, file)

# Print a message showing the model file name.
print("Model saved as heart_disease_prediction_model.pkl")


In [ ]:
# ============================================================
# Step 15: Reflection questions for final showcase
# ============================================================
# These questions help students explain their project to others.
# A good AI project is not only about code.
# Students should also understand the problem, data, model, results, and limitations.

# Create a list of reflection questions.
reflection_questions = [

    # Ask students to explain the project problem.
    "1. What problem does your AI model explore?",

    # Ask students to identify the dataset.
    "2. What dataset did you use?",

    # Ask students to name the model inputs.
    "3. What input features did your model use?",

    # Ask students to explain the prediction output.
    "4. What does the model predict?",

    # Ask students to report the accuracy.
    "5. What was your model accuracy?",

    # Ask students to interpret feature importance.
    "6. Which feature seemed most important?",

    # Ask students to think about model mistakes.
    "7. Did the model make any mistakes?",

    # Ask students to think about responsible AI in healthcare.
    "8. Why should AI predictions in healthcare be used carefully?",

    # Ask students to clearly state the limitation of this class project.
    "9. Why should this model not be used as medical advice?"
]

# Print a heading before the questions.
print("\nReflection Questions:")

# Loop through each question in the list.
for question in reflection_questions:

    # Print the current question.
    print(question)


In [ ]:
# ============================================================
# Presentation Template for Students
# ============================================================
# This cell stores a simple presentation template as text.
# Students can copy this template and fill in their own results.
# The triple quotation marks allow us to write a multi-line string.

# Store the presentation template in a variable.
presentation_template = """
Project Title:
Heart Disease Risk Prediction AI Model

Team Members:
[Names]

Problem:
We wanted to explore how AI can use health-related data to predict possible heart disease risk.

Dataset:
We used the UCI Heart Disease Dataset.
Dataset Link:
https://archive.ics.uci.edu/dataset/45/heart+disease

Inputs:
Age, resting blood pressure, cholesterol, and maximum heart rate.

Prediction:
Lower Risk or Higher Risk

Model Used:
Random Forest Classifier

Result:
Our model reached approximately ____% accuracy.

Demo:
We entered fictional patient values, and the model predicted lower or higher risk.

What We Learned:
We learned how AI can find patterns in healthcare data.

Responsible AI Note:
This project is for education only. It is not medical advice and should not be used for real diagnosis.

Future Improvement:
We could include more medical features, use more data, and involve medical professionals to evaluate the model.
"""

# Print the presentation template so students can see and copy it.
print(presentation_template)
